In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
import tqdm
from pathlib import Path
from models_under_pressure.activation_store import ActivationStore
from models_under_pressure.interfaces.dataset import LabelledDataset
import os

/Users/ep/Documents/research/vrcp_proj/reliable-llm-monitoring/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
LAYER = 11
DATA_DIR = os.environ["DATA_DIR"]

In [5]:
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

device(type='mps')

In [6]:
def batched_average_over_sequence(
    activations: torch.Tensor, 
    attention_mask: torch.Tensor,
    batch_size: int = 256,
    device=device,
) -> torch.Tensor:
    """Compute average over sequence dimension in batches to save memory."""
    n_samples = activations.shape[0]
    averaged_acts = []
    for start_idx in tqdm.tqdm(range(0, n_samples, batch_size)):
        end_idx = min(start_idx + batch_size, n_samples)
        batch_acts = activations[start_idx:end_idx].to(device)  # (batch, seq_len, hidden_dim)
        batch_mask = attention_mask[start_idx:end_idx].to(device)
        masked_acts = batch_acts * batch_mask.unsqueeze(-1)  # (batch, seq_len, hidden_dim)

        sum_acts = masked_acts.sum(dim=1)  # (batch, hidden_dim)
        lengths = batch_mask.sum(dim=1, keepdim=True)  # (batch, 1)
        avg_acts = sum_acts / lengths  # (batch, hidden_dim)
        averaged_acts.append(avg_acts.cpu()) # Move to CPU to save GPU memory

    return torch.cat(averaged_acts, dim=0)  # (n_samples, hidden_dim)

In [7]:
def train_probe(
    dataset: LabelledDataset,
) -> LogisticRegression:
    activations = dataset.other_fields["activations"]  # (n_samples, seq_len, hidden_dim)
    attention_mask = dataset.other_fields["attention_mask"]  # (n_samples, seq_len)
    labels = dataset.labels_numpy() 

    # Compute average activations over sequence dimension
    avg_activations = batched_average_over_sequence(activations, attention_mask)  # (n_samples, hidden_dim)

    # Train logistic regression probe
    probe = LogisticRegression(max_iter=1000)
    probe.fit(avg_activations, labels)

    return probe

In [8]:
import random


def load_dataset(dataset_path: Path, load_activtions: bool) -> LabelledDataset:
    dataset = LabelledDataset.load_from(dataset_path)
    if not load_activtions:
        return dataset
    # Load and attach precomputed activations
    store = ActivationStore()  # uses DATA_DIR/activations via config
    dataset = store.enrich(
        dataset=dataset,
        path=dataset_path,
        model_name=MODEL_NAME,
        layer=LAYER,
        mmap=True,  # Use memory-mapped files for large datasets
    )
    return dataset


def sample_from_dataset(dataset: LabelledDataset, num_samples: int) -> LabelledDataset:
    indices = list(range(len(dataset)))
    sample = random.sample(indices, num_samples)
    return dataset[sample]

# Simple Probe

In [14]:
train_dataset = load_dataset(
    Path(f"{DATA_DIR}/training/prompts_4x/train.jsonl"),
    load_activtions=True,
)
test_dataset = load_dataset(
    Path(f"{DATA_DIR}/training/prompts_4x/test.jsonl"),
    load_activtions=True,
)

# Now dataset_with_acts.other_fields contains:
# - "activations": (batch, seq_len, hidden_dim) tensor
# - "attention_mask": (batch, seq_len)
# - "input_ids": (batch, seq_len)

In [ ]:
train_X = batched_average_over_sequence(
    train_dataset.other_fields["activations"],
    train_dataset.other_fields["attention_mask"],
    batch_size=512,
).numpy()

train_y = train_dataset.labels_numpy()
clf = LogisticRegression(max_iter=1000)
clf.fit(train_X, train_y)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [16]:
test_X = batched_average_over_sequence(
    test_dataset.other_fields["activations"],
    test_dataset.other_fields["attention_mask"],
    batch_size=512,
).numpy()
test_y = test_dataset.labels_numpy()

100%|██████████| 4/4 [00:08<00:00,  2.06s/it]


In [17]:
probs = clf.predict_proba(test_X)[:, 1]
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(test_y, probs)
print(f"Test ROC AUC: {roc_auc:.4f}")

Test ROC AUC: 0.9968


In [18]:
import plotly.express as px
px.histogram(probs, nbins=50, title="Predicted probabilities histogram").show()

In [ ]:
# TODO 
# implement ltt (replace baseline with abstention for now)
# implement naive thresholding method and compare 


# Calling LLM baseline

In [15]:
train_dataset = load_dataset(
    Path(f"{DATA_DIR}/training/prompts_4x/train.jsonl"),
    load_activtions=False,
)[:64]

test_dataset = load_dataset(
    Path(f"{DATA_DIR}/training/prompts_4x/test.jsonl"),
    load_activtions=False,
)

In [16]:
from models_under_pressure.experiments.monitoring_cascade import (
    get_model_baseline_prompt, 
    get_abbreviated_model_name
)
from models_under_pressure.baselines.continuation import likelihood_continuation_prompts

prompt_key = get_model_baseline_prompt(get_abbreviated_model_name(MODEL_NAME))
prompt_config = likelihood_continuation_prompts[prompt_key]

In [20]:
from models_under_pressure.model import LLMModel
from models_under_pressure.config import LOCAL_MODELS
from models_under_pressure.baselines.continuation import LikelihoodContinuationBaseline

model = LLMModel.load(LOCAL_MODELS[get_abbreviated_model_name(MODEL_NAME)])

batch_size = 16
classifier = LikelihoodContinuationBaseline(model, prompt_config)
results = classifier.likelihood_classify_dataset(
    train_dataset,  # type: ignore
    batch_size=batch_size,
)
labels = [label.to_int() for label in list(results.labels)]
ground_truth = train_dataset.labels_numpy()

accuracy = float(np.mean(labels == ground_truth))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
`torch_dtype` is deprecated! Use `dtype` instead!
100%|██████████| 8/8 [00:19<00:00,  2.44s/it]


In [21]:
results.other_fields.keys()

dict_keys(['labels', 'high_stakes_score', 'low_stakes_score', 'high_stakes_log_likelihood', 'low_stakes_log_likelihood', 'model', 'token_counts'])

In [24]:
np.array(results.other_fields['high_stakes_score']) + np.array(results.other_fields['low_stakes_score'])

array([1.00000004, 1.00000003, 1.00000003, 1.        , 1.00000001,
       1.00000003, 0.99999999, 0.99999996, 1.00000004, 1.        ,
       1.00000003, 1.00000001, 1.00000001, 1.00000003, 0.99999994,
       1.00000006, 1.00000001, 1.00000003, 0.99999997, 1.00000003,
       1.00000003, 1.00000006, 1.        , 1.00000001, 1.00000007,
       1.        , 0.99999997, 0.99999997, 1.00000006, 1.00000001,
       1.00000001, 1.00000001, 1.00000001, 1.00000006, 1.        ,
       0.99999997, 1.        , 1.00000003, 1.00000003, 1.        ,
       1.        , 0.99999997, 1.00000001, 1.        , 1.00000003,
       0.99999997, 1.        , 0.99999999, 0.99999998, 0.99999997,
       0.99999997, 0.99999996, 1.        , 0.99999997, 0.99999995,
       1.00000006, 1.00000001, 1.00000006, 0.99999997, 1.00000006,
       0.99999997, 0.99999999, 1.00000004, 1.        ])

# Cascade

In [9]:
from ltt_utils import split_dataset
from dataclasses import dataclass
from models_under_pressure.experiments.monitoring_cascade import (
    get_model_baseline_prompt, 
    get_abbreviated_model_name
)
from models_under_pressure.baselines.continuation import likelihood_continuation_prompts
from models_under_pressure.model import LLMModel
from models_under_pressure.config import LOCAL_MODELS
from models_under_pressure.baselines.continuation import LikelihoodContinuationBaseline


@dataclass
class CascadePredictionResults:
    """ Organises the results of running a cascade of a probe and a baseline model. """
    probe_scores: np.ndarray
    baseline_scores: np.ndarray
    used_baseline: np.ndarray  # Boolean array indicating where the baseline was used
    final_scores: np.ndarray


# TODO refine the type here, really we are happy with anything that returns the logits
def run_cascade(
    probe: callable, 
    baseline_model_name: str, 
    threshold: float, 
    dataset: LabelledDataset,
    baseline_batch_size: int = 16,
    merge_strategy: str = "avg",
) -> CascadePredictionResults:
    """
    Run a cascade of a probe and a baseline model on the given dataset.
    Args:
        probe: A callable that takes in a dataset and returns probe scores (logits).
        baseline_model_name: The name of the baseline model to use (from the McKenzie et al. codebase)
        threshold: The threshold for the probe scores to decide when to use the baseline model.
        dataset: The dataset to run the cascade on.
        merge_strategy: The strategy to merge probe and baseline model scores ("avg" or "replace").
    """
    probe_scores = probe(dataset)
    to_call_baseline = np.logical_and(probe_scores < threshold, 1-probe_scores < threshold)

    prompt_key = get_model_baseline_prompt(get_abbreviated_model_name(MODEL_NAME))
    prompt_config = likelihood_continuation_prompts[prompt_key]
    model = LLMModel.load(LOCAL_MODELS[get_abbreviated_model_name(baseline_model_name)])
    baseline_model = LikelihoodContinuationBaseline(model, prompt_config=prompt_config)

    # Use boolean array to select examples where we need to call the baseline
    baseline_indices = np.where(to_call_baseline)[0].tolist()
    dataset_to_call_baseline = dataset[baseline_indices]
    baseline_results = baseline_model.likelihood_classify_dataset(
        dataset_to_call_baseline, 
        batch_size=baseline_batch_size
    )
    baseline_low_stakes_prob = np.array(baseline_results.other_fields['low_stakes_score'])

    # re-index baseline scores to match the original dataset
    # baseline_scores is nan where baseline was not called
    baseline_scores = np.full_like(probe_scores, np.nan)
    baseline_scores[to_call_baseline] = baseline_low_stakes_prob

    if merge_strategy == "avg":
        final_scores = np.where(
            to_call_baseline, 
            (probe_scores + baseline_scores) / 2,
            probe_scores
        )
    elif merge_strategy == "replace":
        final_scores = np.where(
            to_call_baseline, 
            baseline_scores,
            probe_scores
        )
    else:
        raise ValueError(f"Unknown merge strategy: {merge_strategy}")
    return CascadePredictionResults(
        probe_scores=probe_scores,
        baseline_scores=baseline_scores,
        used_baseline=to_call_baseline,
        final_scores=final_scores,
    )


def probe_function(clf: LogisticRegression, dataset: LabelledDataset) -> np.ndarray:
    activations = dataset.other_fields["activations"]
    attention_mask = dataset.other_fields["attention_mask"]
    X = batched_average_over_sequence(
        activations,
        attention_mask,
        batch_size=512,
    ).numpy()
    probs = clf.predict_proba(X)[:, 1]
    return probs
    


In [10]:
dataset = load_dataset(Path(f"{DATA_DIR}/training/prompts_4x/train.jsonl"), load_activtions=True)
dataset = sample_from_dataset(dataset, 1000)
train, opt, calibration, test = split_dataset(dataset, [0.7, 0.1, 0.1, 0.1], shuffle=True, random_seed=42)

In [11]:
probe = train_probe(train)

100%|██████████| 3/3 [00:00<00:00, 11.50it/s]


In [12]:
cascade_scores = run_cascade(
    probe=lambda dataset: probe_function(probe, dataset),
    baseline_model_name=MODEL_NAME,
    threshold=0.6,
    dataset=dataset,
    merge_strategy="avg",
)
from sklearn.metrics import roc_auc_score

train_probe_roc_auc = roc_auc_score(
    dataset.labels_numpy(),
    cascade_scores.probe_scores,
)
train_final_roc_auc = roc_auc_score(
    dataset.labels_numpy(),
    cascade_scores.final_scores,
)
print(f"Probe ROC AUC: {train_probe_roc_auc:.4f}")
print(f"Cascade ROC AUC: {train_final_roc_auc:.4f}")

100%|██████████| 2/2 [00:00<00:00,  4.88it/s]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
`torch_dtype` is deprecated! Use `dtype` instead!
100%|██████████| 3/3 [00:05<00:00,  1.74s/it]

Probe ROC AUC: 0.9987
Cascade ROC AUC: 0.9985


In [ ]:
def baseline_budget_cost(cascade_scores: CascadePredictionResults) -> float:
    """Rate at which you call the baseline"""
    return cascade_scores.used_baseline.mean()

def empirical_performance(cascade_scores: CascadePredictionResults, dataset: LabelledDataset) -> float:
    """Empirical performance of the cascade."""
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(
        dataset.labels_numpy(),
        cascade_scores.final_scores,
    )

In [13]:
thresholds = np.linspace(0, 1, 11)[1:-1]  # Exclude 0 and 1
budget_scores = []
empirical_performance_scores = []
for i, t in enumerate(thresholds):
    print(f"Evaluating threshold: {t:.2f}, {i} of {len(thresholds)}")
    opt_scores = run_cascade(
        probe=lambda dataset: probe_function(probe, dataset),
        baseline_model_name=MODEL_NAME,
        threshold=t,
        dataset=opt,
        merge_strategy="avg",
    )
    budget_scores.append(baseline_budget_cost(opt_scores))
    empirical_performance_scores.append(empirical_performance(opt_scores, opt))

Evaluating threshold: 0.10, 0 of 9


100%|██████████| 1/1 [00:00<00:00, 25.95it/s]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
`torch_dtype` is deprecated! Use `dtype` instead!
0it [00:00, ?it/s]


Evaluating threshold: 0.20, 1 of 9


100%|██████████| 1/1 [00:00<00:00, 21.07it/s]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
0it [00:00, ?it/s]


Evaluating threshold: 0.30, 2 of 9


...

100%|██████████| 1/1 [00:00<00:00, 24.28it/s]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
100%|██████████| 1/1 [00:02<00:00,  2.52s/it]


Evaluating threshold: 0.80, 7 of 9


100%|██████████| 1/1 [00:00<00:00, 40.67it/s]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
100%|██████████| 2/2 [00:05<00:00,  2.86s/it]


Evaluating threshold: 0.90, 8 of 9


100%|██████████| 1/1 [00:00<00:00,  3.05it/s]
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
100%|██████████| 4/4 [00:11<00:00,  2.81s/it]


In [19]:
costs = np.concatenate(
    [
        -np.array(budget_scores).reshape(-1, 1),   # we want to minimise the budget cost
        np.array(empirical_performance_scores).reshape(-1, 1)
    ],
    axis=1
)

from ltt_utils import is_pareto
pareto_front = is_pareto(costs)
pareto_front

array([False, False, False, False, False, False, False, False,  True])

In [23]:
import plotly.express as px
fig = px.scatter(
    x=-costs[:, 0],  # Budget cost (negative because we minimized -cost)
    y=costs[:, 1],   # Empirical performance
    title="Budget Cost vs Empirical Performance",
    labels={"x": "Budget Cost", "y": "Empirical Performance"},
)
# fig.add_scatter(
#     x=-costs[pareto_front, 0],
#     y=costs[pareto_front, 1],
#     mode='markers',
#     marker=dict(color='red', size=10, symbol='star'),
#     name='Pareto Front'
# )
fig.show()

In [20]:
budget_scores

[0.0, 0.0, 0.0, 0.0, 0.0, 0.02, 0.06, 0.15, 0.31]

In [21]:
empirical_performance_scores

[0.9779911964785915,
 0.9779911964785915,
 0.9779911964785915,
 0.9779911964785915,
 0.9779911964785915,
 0.9771908763505402,
 0.9771908763505402,
 0.9735894357743098,
 0.9679871948779513]

In [ ]:
# run this on a more realistic split, and intergate it with probe training properly

In [ ]:
# now define the empirical risk functions that we will use to find the pareto front and run LTT

# LTT

In [ ]:
# given train, validation, calibration and test splits 
# train probes on `train`
# e2e score pipeline: give validation dataset and threshold, get final performance metrics (cost + score)
# vanilla threshold estimation using calibration set once
# LTT using above e2e pipeline and calibration set + pareto frontier etc...
# comapre e2e performance distributions across multiple runs

In [ ]:
# make e2e pipeline with final performance score + cost as a function of dataset splits
# find pareto front
# run fixed sequence testing